# Chapter 14 — Intro to LSTMs: Learn to Write Like Shakespeare

**Mata Kuliah:** Deep Learning
**Buku:** Grokking Deep Learning (Andrew W. Trask, Manning, 2019)
**Chapter 14:** Intro to LSTMs - Learn to Write Like Shakespeare

## Isi Chapter 14

1. Character language modeling
2. Truncated backpropagation
3. Menghasilkan teks dari model (sampling)
4. Vanishing dan exploding gradients
5. Contoh kecil backpropagation pada RNN
6. LSTM: gated copy trick dan intuisi gate
7. Mengganti RNN dengan LSTM pada model karakter
8. Uji tambahan: memeriksa gradien framework

## Catatan sebelum menjalankan

- **Data:** `shakespear.txt` (karya Shakespeare, sekitar 100.000 karakter) diunduh otomatis
  dari repository resmi buku. Jalankan di Google Colab.
- **Waktu:** training model karakter berat untuk NumPy murni dengan lapisan 512 unit.
  Satu epoch RNN sekitar satu menit di mesin saya dan satu epoch LSTM beberapa menit; di
  Colab bisa lebih lama. Jumlah epoch di bawah sengaja kecil (`EPOCHS_RNN`,
  `EPOCHS_LSTM`) supaya notebook bisa dijalankan sampai selesai. Model dengan kualitas
  teks yang bagus membutuhkan jauh lebih banyak epoch (jam, bukan menit).
- **Framework:** chapter ini memakai framework mini dari Chapter 13. Karena notebook
  terpisah, kodenya saya tulis ulang di bawah dengan beberapa perubahan kecil yang
  ditandai di komentar kode.

## 1. Character language modeling

Di Chapter 12 dan 13, RNN dilatih pada dataset buatan yang dihasilkan dengan aturan
sederhana. Sekarang tugasnya lebih sulit: memodelkan bahasa pada karya Shakespeare, dan
unit yang diprediksi adalah karakter, bukan kata. Model membaca karakter-karakter yang
sudah dilihat lalu memprediksi karakter berikutnya. Vocabulary-nya menjadi kumpulan
karakter dalam dataset, dan teksnya diubah menjadi array indeks karakter.

In [ ]:
import os, urllib.request

RAW = 'https://raw.githubusercontent.com/iamtrask/Grokking-Deep-Learning/{}/{}'

def ensure_file(name):
    if os.path.exists(name):
        return
    for branch in ['master', 'main']:          # cabang repo bisa master atau main
        try:
            urllib.request.urlretrieve(RAW.format(branch, name), name)
            return
        except Exception:
            pass
    raise RuntimeError('Gagal mengunduh ' + name)

ensure_file('shakespear.txt')

Berikutnya framework dari Chapter 13. Isinya kelas `Tensor` dengan autograd, lalu layer
dan optimizer. Perubahan dari versi Chapter 13:

1. `Tensor.softmax()` ditambahkan (dipakai untuk menghasilkan teks).
2. `Linear` punya parameter `bias=True`, agar bisa dibuat tanpa bias (dipakai LSTM).
3. Rentang `id` acak diperbesar menjadi satu miliar. Graph di chapter ini berisi ribuan
   tensor, jadi dengan rentang 100.000 tabrakan id lebih mungkin terjadi.
4. Pada `backward`, `raise Exception("cannot backprop more than once")` diganti `return`.
   Tanpa perubahan ini, LSTM berhenti dengan error tersebut saat `loss.backward()`.
   Konsekuensinya saya bahas di bagian terakhir notebook.

In [ ]:
import numpy as np

class Tensor (object):

    def __init__(self,data,
                 autograd=False,
                 creators=None,
                 creation_op=None,
                 id=None):

        self.data = np.array(data)
        self.autograd = autograd
        self.grad = None
        if(id is None):
            self.id = np.random.randint(0,1000000000)   # berbeda dari Chapter 13: rentang id diperbesar (sebelumnya 100000)
        else:
            self.id = id

        self.creators = creators
        self.creation_op = creation_op
        self.children = {}

        if(creators is not None):
            for c in creators:
                if(self.id not in c.children):
                    c.children[self.id] = 1
                else:
                    c.children[self.id] += 1

    def all_children_grads_accounted_for(self):
        for id,cnt in self.children.items():
            if(cnt != 0):
                return False
        return True

    def backward(self,grad=None, grad_origin=None):
        if(self.autograd):

            if(grad is None):
                grad = Tensor(np.ones_like(self.data))

            if(grad_origin is not None):
                if(self.children[grad_origin.id] == 0):
                    return   # berbeda dari Chapter 13: sebelumnya raise Exception("cannot backprop more than once")
                else:
                    self.children[grad_origin.id] -= 1

            if(self.grad is None):
                self.grad = grad
            else:
                self.grad += grad

            # grads must not have grads of their own
            assert grad.autograd == False

            # only continue backpropping if there's something to
            # backprop into and if all gradients (from children)
            # are accounted for override waiting for children if
            # "backprop" was called on this variable directly
            if(self.creators is not None and
               (self.all_children_grads_accounted_for() or
                grad_origin is None)):

                if(self.creation_op == "add"):
                    self.creators[0].backward(self.grad, self)
                    self.creators[1].backward(self.grad, self)

                if(self.creation_op == "sub"):
                    self.creators[0].backward(Tensor(self.grad.data), self)
                    self.creators[1].backward(Tensor(self.grad.__neg__().data), self)

                if(self.creation_op == "mul"):
                    new = self.grad * self.creators[1]
                    self.creators[0].backward(new , self)
                    new = self.grad * self.creators[0]
                    self.creators[1].backward(new, self)

                if(self.creation_op == "mm"):
                    c0 = self.creators[0]
                    c1 = self.creators[1]
                    new = self.grad.mm(c1.transpose())
                    c0.backward(new)
                    new = self.grad.transpose().mm(c0).transpose()
                    c1.backward(new)

                if(self.creation_op == "transpose"):
                    self.creators[0].backward(self.grad.transpose())

                if("sum" in self.creation_op):
                    dim = int(self.creation_op.split("_")[1])
                    self.creators[0].backward(self.grad.expand(dim,
                                                               self.creators[0].data.shape[dim]))

                if("expand" in self.creation_op):
                    dim = int(self.creation_op.split("_")[1])
                    self.creators[0].backward(self.grad.sum(dim))

                if(self.creation_op == "neg"):
                    self.creators[0].backward(self.grad.__neg__())

                if(self.creation_op == "sigmoid"):
                    ones = Tensor(np.ones_like(self.grad.data))
                    self.creators[0].backward(self.grad * (self * (ones - self)))

                if(self.creation_op == "tanh"):
                    ones = Tensor(np.ones_like(self.grad.data))
                    self.creators[0].backward(self.grad * (ones - (self * self)))

                if(self.creation_op == "index_select"):
                    new_grad = np.zeros_like(self.creators[0].data)
                    indices_ = self.index_select_indices.data.flatten()
                    grad_ = grad.data.reshape(len(indices_), -1)
                    for i in range(len(indices_)):
                        new_grad[indices_[i]] += grad_[i]
                    self.creators[0].backward(Tensor(new_grad))

                if(self.creation_op == "cross_entropy"):
                    dx = self.softmax_output - self.target_dist
                    self.creators[0].backward(Tensor(dx))

    def __add__(self, other):
        if(self.autograd and other.autograd):
            return Tensor(self.data + other.data,
                          autograd=True,
                          creators=[self,other],
                          creation_op="add")
        return Tensor(self.data + other.data)

    def __neg__(self):
        if(self.autograd):
            return Tensor(self.data * -1,
                          autograd=True,
                          creators=[self],
                          creation_op="neg")
        return Tensor(self.data * -1)

    def __sub__(self, other):
        if(self.autograd and other.autograd):
            return Tensor(self.data - other.data,
                          autograd=True,
                          creators=[self,other],
                          creation_op="sub")
        return Tensor(self.data - other.data)

    def __mul__(self, other):
        if(self.autograd and other.autograd):
            return Tensor(self.data * other.data,
                          autograd=True,
                          creators=[self,other],
                          creation_op="mul")
        return Tensor(self.data * other.data)

    def sum(self, dim):
        if(self.autograd):
            return Tensor(self.data.sum(dim),
                          autograd=True,
                          creators=[self],
                          creation_op="sum_"+str(dim))
        return Tensor(self.data.sum(dim))

    def expand(self, dim,copies):

        trans_cmd = list(range(0,len(self.data.shape)))
        trans_cmd.insert(dim,len(self.data.shape))
        new_data = self.data.repeat(copies).reshape(list(self.data.shape) + [copies]).transpose(trans_cmd)

        if(self.autograd):
            return Tensor(new_data,
                          autograd=True,
                          creators=[self],
                          creation_op="expand_"+str(dim))
        return Tensor(new_data)

    def transpose(self):
        if(self.autograd):
            return Tensor(self.data.transpose(),
                          autograd=True,
                          creators=[self],
                          creation_op="transpose")

        return Tensor(self.data.transpose())

    def mm(self, x):
        if(self.autograd):
            return Tensor(self.data.dot(x.data),
                          autograd=True,
                          creators=[self,x],
                          creation_op="mm")
        return Tensor(self.data.dot(x.data))

    def sigmoid(self):
        if(self.autograd):
            return Tensor(1 / (1 + np.exp(-self.data)),
                          autograd=True,
                          creators=[self],
                          creation_op="sigmoid")
        return Tensor(1 / (1 + np.exp(-self.data)))

    def tanh(self):
        if(self.autograd):
            return Tensor(np.tanh(self.data),
                          autograd=True,
                          creators=[self],
                          creation_op="tanh")
        return Tensor(np.tanh(self.data))

    def index_select(self, indices):

        if(self.autograd):
            new = Tensor(self.data[indices.data],
                         autograd=True,
                         creators=[self],
                         creation_op="index_select")
            new.index_select_indices = indices
            return new
        return Tensor(self.data[indices.data])

    def softmax(self):
        temp = np.exp(self.data)
        softmax_output = temp / np.sum(temp,
                                       axis=len(self.data.shape)-1,
                                       keepdims=True)
        return softmax_output

    def cross_entropy(self, target_indices):

        temp = np.exp(self.data)
        softmax_output = temp / np.sum(temp,
                                       axis=len(self.data.shape)-1,
                                       keepdims=True)

        t = target_indices.data.flatten()
        p = softmax_output.reshape(len(t),-1)
        target_dist = np.eye(p.shape[1])[t]
        loss = -(np.log(p) * (target_dist)).sum(1).mean()

        if(self.autograd):
            out = Tensor(loss,
                         autograd=True,
                         creators=[self],
                         creation_op="cross_entropy")
            out.softmax_output = softmax_output
            out.target_dist = target_dist
            return out

        return Tensor(loss)

    def __repr__(self):
        return str(self.data.__repr__())

    def __str__(self):
        return str(self.data.__str__())

In [ ]:
class Layer(object):

    def __init__(self):
        self.parameters = list()

    def get_parameters(self):
        return self.parameters


class SGD(object):

    def __init__(self, parameters, alpha=0.1):
        self.parameters = parameters
        self.alpha = alpha

    def zero(self):
        for p in self.parameters:
            p.grad.data *= 0

    def step(self, zero=True):

        for p in self.parameters:

            p.data -= p.grad.data * self.alpha

            if(zero):
                p.grad.data *= 0


class Linear(Layer):

    def __init__(self, n_inputs, n_outputs, bias=True):
        super().__init__()

        self.use_bias = bias

        W = np.random.randn(n_inputs, n_outputs) * np.sqrt(2.0/(n_inputs))
        self.weight = Tensor(W, autograd=True)
        if(self.use_bias):
            self.bias = Tensor(np.zeros(n_outputs), autograd=True)

        self.parameters.append(self.weight)

        if(self.use_bias):
            self.parameters.append(self.bias)

    def forward(self, input):
        if(self.use_bias):
            return input.mm(self.weight)+self.bias.expand(0,len(input.data))
        return input.mm(self.weight)


class Embedding(Layer):

    def __init__(self, vocab_size, dim):
        super().__init__()

        self.vocab_size = vocab_size
        self.dim = dim

        # this random initialiation style is just a convention from word2vec
        self.weight = Tensor((np.random.rand(vocab_size, dim) - 0.5) / dim, autograd=True)

        self.parameters.append(self.weight)

    def forward(self, input):
        return self.weight.index_select(input)


class Tanh(Layer):
    def __init__(self):
        super().__init__()

    def forward(self, input):
        return input.tanh()


class Sigmoid(Layer):
    def __init__(self):
        super().__init__()

    def forward(self, input):
        return input.sigmoid()


class CrossEntropyLoss(object):

    def __init__(self):
        super().__init__()

    def forward(self, input, target):
        return input.cross_entropy(target)


class RNNCell(Layer):

    def __init__(self, n_inputs, n_hidden, n_output, activation='sigmoid'):
        super().__init__()

        self.n_inputs = n_inputs
        self.n_hidden = n_hidden
        self.n_output = n_output

        if(activation == 'sigmoid'):
            self.activation = Sigmoid()
        elif(activation == 'tanh'):
            self.activation == Tanh()
        else:
            raise Exception("Non-linearity not found")

        self.w_ih = Linear(n_inputs, n_hidden)
        self.w_hh = Linear(n_hidden, n_hidden)
        self.w_ho = Linear(n_hidden, n_output)

        self.parameters += self.w_ih.get_parameters()
        self.parameters += self.w_hh.get_parameters()
        self.parameters += self.w_ho.get_parameters()

    def forward(self, input, hidden):
        from_prev_hidden = self.w_hh.forward(hidden)
        combined = self.w_ih.forward(input) + from_prev_hidden
        new_hidden = self.activation.forward(combined)
        output = self.w_ho.forward(new_hidden)
        return output, new_hidden

    def init_hidden(self, batch_size=1):
        return Tensor(np.zeros((batch_size,self.n_hidden)), autograd=True)

Teks dan indeksnya. Vocabulary saya urutkan (`sorted`) agar indeks karakter sama setiap
kali dijalankan; `list(set(raw))` urutannya bisa berubah antar sesi.

In [ ]:
import sys,random,math
from collections import Counter
import numpy as np
import sys

np.random.seed(0)

# dataset from http://karpathy.github.io/2015/05/21/rnn-effectiveness/
f = open('shakespear.txt','r')
raw = f.read()
f.close()

vocab = sorted(set(raw))
word2index = {}
for i,word in enumerate(vocab):
    word2index[word]=i
indices = np.array(list(map(lambda x:word2index[x], raw)))

print('jumlah karakter :', len(raw))
print('ukuran vocabulary:', len(vocab))
print(repr(raw[0:5]))
print(indices[0:5])

Dataset berisi 99.993 karakter dengan 62 karakter unik (huruf, tanda baca, spasi, baris baru). Lima karakter pertama adalah `That,` dan indeksnya `[29 43 36 55  4]`.

## 2. Perlunya truncated backpropagation

Di Chapter 13, RNN membaca lima kata lalu memprediksi kata keenam, dan
`loss.backward()` merambatkan gradien sampai ke input pertama. Itu mungkin karena
datanya pendek (tidak ada contoh yang lebih panjang dari enam kata). Pada Shakespeare
ada sekitar 100.000 karakter, dan backpropagation melalui semuanya untuk setiap
prediksi tidak mungkin dilakukan.

Solusinya: backpropagate hanya sejumlah langkah tetap ke belakang, lalu berhenti. Ini
disebut *truncated backpropagation*, dan menjadi standar industri. Panjangnya menjadi
parameter yang bisa diatur seperti batch size atau alpha. Untuk language modeling
namanya `bptt`, biasanya antara 16 dan 64.

Kelemahannya, jaringan tidak bisa belajar mengingat kejadian yang lebih jauh dari
`bptt` langkah ke belakang. Secara tegas, bisa saja ada sisa informasi dari lebih
jauh di hidden state, tetapi gradien tidak bisa meminta jaringan menyimpannya. Kelemahan
lain: logika mini-batch menjadi lebih rumit.

Cara menyusun data: anggap datasetnya adalah banyak dataset kecil berukuran `bptt`.
Pertama data dipotong agar kelipatan `batch_size`, dibentuk menjadi matriks
`(batch_size, n_batches)` lalu ditranspos, sehingga setiap kolom adalah potongan
berurutan dari teks. Target adalah input yang digeser satu baris (memprediksi karakter
berikutnya). Terakhir dibentuk `input_batches` dan `target_batches` berbentuk
`(n_bptt, bptt, batch_size)`.

Untuk melihatnya jelas, saya tampilkan dulu dengan `batch_size = 8`:

In [ ]:
batch_size = 8
bptt = 16
n_batches = int((indices.shape[0] / (batch_size)))

trimmed_indices = indices[:n_batches*batch_size]
batched_indices = trimmed_indices.reshape(batch_size, n_batches).transpose()

input_batched_indices = batched_indices[0:-1]
target_batched_indices = batched_indices[1:]

n_bptt = int(((n_batches-1) / bptt))
input_batches = input_batched_indices[:n_bptt*bptt].reshape(n_bptt,bptt,batch_size)
target_batches = target_batched_indices[:n_bptt*bptt].reshape(n_bptt, bptt, batch_size)

print(raw[0:5])
print(indices[0:5])
print()
print(batched_indices[0:5])
print()
print(input_batches[0][0:5])
print()
print(target_batches[0][0:5])
print()
print('bentuk input_batches:', input_batches.shape)

Setiap kolom pada `batched_indices` adalah potongan teks yang berurutan: kolom pertama dimulai dari `That,`, sedangkan kolom lain dimulai dari titik yang jauh di dalam teks. `target_batches` sama dengan `input_batches` yang digeser satu baris ke atas, jadi untuk setiap karakter targetnya adalah karakter berikutnya. Dengan `batch_size = 8` dan `bptt = 16` terbentuk 781 potongan (`input_batches` berbentuk `(781, 16, 8)`). Pada pengaturan sebenarnya (`batch_size = 32`) ada 195 potongan per epoch.

Sekarang pengaturan sebenarnya untuk model RNN: embedding dan hidden berukuran 512,
`batch_size = 32`, `bptt = 16`.

In [ ]:
embed = Embedding(vocab_size=len(vocab),dim=512)
model = RNNCell(n_inputs=512, n_hidden=512, n_output=len(vocab))

criterion = CrossEntropyLoss()
optim = SGD(parameters=model.get_parameters() + embed.get_parameters(), alpha=0.05)

batch_size = 32
bptt = 16
n_batches = int((indices.shape[0] / (batch_size)))

trimmed_indices = indices[:n_batches*batch_size]
batched_indices = trimmed_indices.reshape(batch_size, n_batches).transpose()

input_batched_indices = batched_indices[0:-1]
target_batched_indices = batched_indices[1:]

n_bptt = int(((n_batches-1) / bptt))
input_batches = input_batched_indices[:n_bptt*bptt].reshape(n_bptt,bptt,batch_size)
target_batches = target_batched_indices[:n_bptt*bptt].reshape(n_bptt, bptt, batch_size)

print('jumlah batch per epoch:', len(input_batches))

## 3. Menghasilkan teks (sampling)

Untuk melihat apa yang dipelajari model, saya buat fungsi `generate_sample`. Fungsi ini
memberi satu karakter awal, lalu berulang kali: memprediksi distribusi karakter
berikutnya, memilih satu karakter, dan memasukkannya lagi sebagai input. Output
dikalikan 10 sebelum softmax; makin besar pengalinya, makin tajam distribusinya
(makin "serakah", condong ke karakter paling mungkin). Pemilihan karakternya memakai
`(temp_dist > np.random.rand()).argmax()`, yaitu indeks pertama yang probabilitasnya
melebihi bilangan acak (jika tidak ada yang melebihi, hasilnya indeks 0).

In [ ]:
def generate_sample(n=30, init_char=' '):
    s = ""
    hidden = model.init_hidden(batch_size=1)
    input = Tensor(np.array([word2index[init_char]]))
    for i in range(n):
        rnn_input = embed.forward(input)
        output, hidden = model.forward(input=rnn_input, hidden=hidden)
        output.data *= 10
        temp_dist = output.softmax()
        temp_dist /= temp_dist.sum()

        m = (temp_dist > np.random.rand()).argmax()
        c = vocab[m]
        input = Tensor(np.array([m]))
        s += c
    return s

## Training dengan truncated backpropagation

Kode training mirip Chapter 13. Bedanya: pada setiap potongan `bptt` langkah, loss
dihitung di tiap langkah, lalu setelah satu potongan selesai dilakukan backpropagation
dan update bobot. Setelah itu pembacaan data berlanjut seolah tidak terjadi apa-apa, dengan
hidden state yang sama dari potongan sebelumnya. Hidden state hanya di-reset di awal setiap
epoch. Baris `Tensor(hidden.data, autograd=True)` membuat tensor baru dari nilai
hidden tanpa riwayat graph, sehingga gradien berhenti di batas potongan (itu bentuk
"truncated"-nya). Setelah satu epoch, `alpha` dikali 0.99.

Satu hal yang saya perhatikan pada kode ini: ada loop `for loss in losses: ""` yang
tidak melakukan apa-apa selain menimpa variabel `loss` dengan elemen terakhir dari
`losses`. Jadi `loss.backward()` dimulai dari loss langkah terakhir saja (bukan jumlah
semua langkah), dan angka loss yang dicetak juga loss langkah terakhir. Gradien tetap
mengalir mundur lewat hidden state ke langkah-langkah sebelumnya. Kodenya saya tulis
apa adanya.

In [ ]:
EPOCHS_RNN = 3      # naikkan untuk hasil yang lebih baik (waktu training bertambah)

def train(iterations=100):
    for iter in range(iterations):
        total_loss = 0
        n_loss = 0

        hidden = model.init_hidden(batch_size=batch_size)
        for batch_i in range(len(input_batches)):

            hidden = Tensor(hidden.data, autograd=True)
            loss = None
            losses = list()
            for t in range(bptt):
                input = Tensor(input_batches[batch_i][t], autograd=True)
                rnn_input = embed.forward(input=input)
                output, hidden = model.forward(input=rnn_input, hidden=hidden)

                target = Tensor(target_batches[batch_i][t], autograd=True)
                batch_loss = criterion.forward(output, target)
                losses.append(batch_loss)
                if(t == 0):
                    loss = batch_loss
                else:
                    loss = loss + batch_loss
            for loss in losses:
                ""
            loss.backward()
            optim.step()
            total_loss += loss.data
            log = "\r Iter:" + str(iter)
            log += " - Batch "+str(batch_i+1)+"/"+str(len(input_batches))
            log += " - Loss:" + str(np.exp(total_loss / (batch_i+1)))
            if(batch_i == 0):
                log += " - " + generate_sample(n=70, init_char='\n').replace("\n"," ")
            if(batch_i % 10 == 0 or batch_i-1 == len(input_batches)):
                sys.stdout.write(log)
        optim.alpha *= 0.99
        print()

train(EPOCHS_RNN)

Loss yang tercetak adalah `exp` dari rata-rata loss langkah terakhir (lihat catatan `for loss in losses`), jadi nilai acak untuk 62 karakter adalah sekitar 62. Awal training tidak stabil: epoch 0 dimulai di sekitar 70, lalu melonjak sampai orde 10^11 pada batch ke-11 dan baru turun pelan, dan masih 5207 di batch ke-191. Epoch 1 berakhir di sekitar 30 dan epoch 2 di sekitar 18,6. Loss turun, tetapi ketidakstabilan awal ini sesuai dengan masalah gradien yang dibahas di bawah. Contoh teks yang dicetak di batch pertama tiap epoch masih berupa pengulangan (`thttttt...`, `that that that ...`).

## Contoh teks dari model RNN

Teks yang dihasilkan memperlihatkan seberapa banyak yang sudah dipelajari. Dengan
`n=2000` dan karakter awal baris baru, model yang sudah dilatih lama bisa menghasilkan
teks mirip naskah (nama tokoh dengan titik dua, baris pendek). Di sini saya ambil 300
karakter dari model yang baru dilatih beberapa epoch:

In [ ]:
print(generate_sample(n=300, init_char='\n'))

Setelah tiga epoch model hanya menghasilkan potongan kata Inggris pendek yang berulang (`ig the he he`) lalu baris kosong. Ia belum belajar struktur naskah. Itu wajar: tiga epoch sangat sedikit untuk lapisan 512 unit.

## 4. Vanishing dan exploding gradients

RNN vanilla menderita gradien yang menghilang (*vanishing*) atau meledak (*exploding*).
Pada forward propagation, hidden state dibentuk dalam tiga langkah: kalikan hidden
sebelumnya dengan matriks bobot, tambahkan embedding kata berikutnya, lalu terapkan
nonlinearitas. Nonlinearitas inilah yang menjaga kestabilan: berapa pun panjang
sekuensnya, hidden state tetap berada di rentang nonlinearitas (0 sampai 1 untuk
sigmoid). Backpropagation tidak punya sifat itu; ia cenderung menghasilkan nilai yang
sangat besar atau sangat kecil. Nilai besar membuat divergensi (banyak NaN), nilai sangat
kecil membuat jaringan tidak belajar.

## 5. Contoh kecil backpropagation pada RNN

Untuk melihatnya langsung, saya jalankan loop backpropagation rekuren dengan aktivasi
sigmoid dan relu pada matriks bobot yang sama. Gradien jadi sangat kecil pada sigmoid
karena turunan sigmoid hampir datar di ujungnya (mendekati 0 atau 1), sedangkan pada
relu gradien tidak diredam sehingga perkalian matriks berulang membuatnya meledak.

In [ ]:
(sigmoid,relu)=(lambda x:1/(1+np.exp(-x)), lambda x:(x>0).astype(float)*x)

weights = np.array([[1,4],[4,1]])
activation = sigmoid(np.array([1,0.01]))

print("Sigmoid Activations")
activations = list()
for iter in range(10):
    activation = sigmoid(activation.dot(weights))
    activations.append(activation)
    print(activation)

print("\nSigmoid Gradients")
gradient = np.ones_like(activation)
for activation in reversed(activations):
    gradient = (activation * (1 - activation) * gradient)
    gradient = gradient.dot(weights.transpose())
    print(gradient)

print("Activations")
activations = list()
for iter in range(10):
    activation = relu(activation.dot(weights))
    activations.append(activation)
    print(activation)

print("\nGradients")
gradient = np.ones_like(activation)
for activation in reversed(activations):
    gradient = ((activation > 0) * gradient).dot(weights.transpose())
    print(gradient)

Pada sigmoid, aktivasi cepat jenuh di sekitar 0,993 dan gradien mengecil sekitar 30 kali lipat tiap langkah mundur: dari 0,034 sampai 1,5e-14 setelah sepuluh langkah. Itu *vanishing gradient*. Pada relu, aktivasi dan gradien malah dikali sekitar 5 tiap langkah: aktivasi mencapai 9,3 juta dan gradien 9.765.625 (5^10) setelah sepuluh langkah. Itu *exploding gradient*. Matriks bobot dan jumlah langkahnya sama; hanya nonlinearitasnya yang berbeda.

## 6. LSTM

LSTM adalah model standar untuk melawan vanishing dan exploding gradients. Masalahnya
ada pada cara hidden state dibentuk: kombinasi perkalian matriks dan nonlinearitas.
Solusi LSTM sederhana.

### Gated copy trick

LSTM membentuk hidden state berikutnya dengan menyalin hidden state sebelumnya lalu
menambah atau menghapus informasi seperlunya. Mekanisme tambah dan hapus itu disebut
*gate*. LSTM punya dua vektor state: `h` (hidden) dan `cell` (c). Yang terpenting adalah
`c`. Pembaruannya:

- `c = f * prev_cell + i * g`
- `f` adalah forget gate: jika bernilai 0, cell sebelumnya dihapus.
- `i` adalah input gate: jika bernilai 1, `g` (vektor pembaruan) ditambahkan penuh ke cell.
- `o` adalah output gate yang mengatur seberapa banyak isi cell boleh dilihat oleh
  prediksi: `h = o * tanh(c)`. Jika `o` semua nol, prediksi mengabaikan cell sama sekali.

Semua gate dibentuk dengan cara yang sama: masing-masing punya matriks bobot sendiri,
bergantung pada input dan hidden sebelumnya, lalu dilewatkan sigmoid. Sigmoid cocok
untuk gate karena jenuh di 0 dan 1.

### Intuisi gate

Gate mirip membaca dan menulis memori. Pembaruan `c` tidak melewati perkalian matriks
maupun nonlinearitas (tidak pernah memanggil `nonlinearity(c)` atau `c.dot(weights)`),
hanya salinan (jika `f` tidak nol) ditambah pembaruan (jika `i` tidak nol). Inilah yang
membuat LSTM bisa menyimpan informasi sepanjang deret waktu tanpa terkena vanishing atau
exploding gradients.

Bagaimana dengan `h`? `h` masih bisa mengalami vanishing gradient, karena dipakai mirip RNN
biasa. Tetapi `h` dibentuk dari vektor yang dihimpit tanh dan sigmoid, sehingga tidak
meledak, hanya bisa menghilang. Itu tidak masalah karena `h` bergantung pada `c`, dan
`c` membawa informasi jarak jauh. Seluruh informasi jarak jauh dibawa oleh `c`, sedangkan `h`
hanyalah tafsiran lokal dari `c` untuk membuat prediksi dan gate di langkah berikutnya.

### Layer LSTM

`LSTMCell` dibangun dengan autograd dari Chapter 13. Ada empat layer dari input
(`xf`, `xi`, `xo`, `xc`), empat layer dari hidden sebelumnya tanpa bias (`hf`, `hi`,
`ho`, `hc`), dan satu layer output `w_ho`. Di kode, vektor pembaruan diberi nama `g`.
`init_hidden` mengembalikan pasangan `(h, c)`; elemen pertama keduanya diisi 1 (detail
kode ini tidak saya uji pengaruhnya).

In [ ]:
class LSTMCell(Layer):

    def __init__(self, n_inputs, n_hidden, n_output):
        super().__init__()

        self.n_inputs = n_inputs
        self.n_hidden = n_hidden
        self.n_output = n_output

        self.xf = Linear(n_inputs, n_hidden)
        self.xi = Linear(n_inputs, n_hidden)
        self.xo = Linear(n_inputs, n_hidden)
        self.xc = Linear(n_inputs, n_hidden)

        self.hf = Linear(n_hidden, n_hidden, bias=False)
        self.hi = Linear(n_hidden, n_hidden, bias=False)
        self.ho = Linear(n_hidden, n_hidden, bias=False)
        self.hc = Linear(n_hidden, n_hidden, bias=False)

        self.w_ho = Linear(n_hidden, n_output, bias=False)

        self.parameters += self.xf.get_parameters()
        self.parameters += self.xi.get_parameters()
        self.parameters += self.xo.get_parameters()
        self.parameters += self.xc.get_parameters()

        self.parameters += self.hf.get_parameters()
        self.parameters += self.hi.get_parameters()
        self.parameters += self.ho.get_parameters()
        self.parameters += self.hc.get_parameters()

        self.parameters += self.w_ho.get_parameters()

    def forward(self, input, hidden):

        prev_hidden = hidden[0]
        prev_cell = hidden[1]

        f = (self.xf.forward(input) + self.hf.forward(prev_hidden)).sigmoid()
        i = (self.xi.forward(input) + self.hi.forward(prev_hidden)).sigmoid()
        o = (self.xo.forward(input) + self.ho.forward(prev_hidden)).sigmoid()
        g = (self.xc.forward(input) + self.hc.forward(prev_hidden)).tanh()
        c = (f * prev_cell) + (i * g)

        h = o * c.tanh()

        output = self.w_ho.forward(h)
        return output, (h, c)

    def init_hidden(self, batch_size=1):
        init_hidden = Tensor(np.zeros((batch_size,self.n_hidden)), autograd=True)
        init_cell = Tensor(np.zeros((batch_size,self.n_hidden)), autograd=True)
        init_hidden.data[:,0] += 1
        init_cell.data[:,0] += 1
        return (init_hidden, init_cell)

## 7. Mengganti RNN dengan LSTM pada model karakter

Berkat framework, hampir tidak ada yang berubah pada pengaturan jaringan. Perbedaannya:

- model memakai `LSTMCell`, dan bobot output diisi nol (`model.w_ho.weight.data *= 0`),
- `batch_size = 16`, `bptt = 25`,
- `min_loss` menyimpan loss terbaik untuk pencatatan.

Data dan vocabulary sama.

In [ ]:
np.random.seed(0)

embed = Embedding(vocab_size=len(vocab),dim=512)
model = LSTMCell(n_inputs=512, n_hidden=512, n_output=len(vocab))
model.w_ho.weight.data *= 0

criterion = CrossEntropyLoss()
optim = SGD(parameters=model.get_parameters() + embed.get_parameters(), alpha=0.05)

batch_size = 16
bptt = 25
n_batches = int((indices.shape[0] / (batch_size)))

trimmed_indices = indices[:n_batches*batch_size]
batched_indices = trimmed_indices.reshape(batch_size, n_batches).transpose()

input_batched_indices = batched_indices[0:-1]
target_batched_indices = batched_indices[1:]

n_bptt = int(((n_batches-1) / bptt))
input_batches = input_batched_indices[:n_bptt*bptt].reshape(n_bptt,bptt,batch_size)
target_batches = target_batched_indices[:n_bptt*bptt].reshape(n_bptt, bptt, batch_size)
min_loss = 1000

print('jumlah batch per epoch:', len(input_batches))

Untuk menghasilkan teks dari LSTM, `generate_sample` yang dipakai mengambil karakter
dengan probabilitas tertinggi (`argmax`) setelah output dikalikan 15. Baris sampling acak
dibiarkan tidak dipakai.

In [ ]:
def generate_sample(n=30, init_char=' '):
    s = ""
    hidden = model.init_hidden(batch_size=1)
    input = Tensor(np.array([word2index[init_char]]))
    for i in range(n):
        rnn_input = embed.forward(input)
        output, hidden = model.forward(input=rnn_input, hidden=hidden)
        output.data *= 15
        temp_dist = output.softmax()
        temp_dist /= temp_dist.sum()

        m = output.data.argmax() # take the max prediction
        c = vocab[m]
        input = Tensor(np.array([m]))
        s += c
    return s

### Training LSTM

Logika training hampir sama dengan RNN. Perubahan yang berarti hanya pada truncated
backpropagation, karena sekarang ada dua vektor hidden per langkah (`h` dan `c`), jadi
keduanya dibungkus ulang menjadi tensor baru di awal setiap potongan. Loss juga dijumlahkan
sepanjang langkah: `losses` menyimpan jumlah kumulatif, dan `loss = losses[-1]` adalah
jumlah seluruh langkah (berbeda dari RNN di atas). Ada tambahan pencatatan:
`alpha` turun pelan tiap epoch (dikali 0.99), dan log menampilkan loss terkecil
(`min_loss`) serta contoh teks.

`global min_loss` ditambahkan karena kode dibungkus dalam fungsi. Tanpa baris itu,
`min_loss` dianggap variabel lokal dan Python akan error.

In [ ]:
EPOCHS_LSTM = 2     # naikkan untuk hasil yang lebih baik (waktu training bertambah)

def train(iterations=400):
    global min_loss
    for iter in range(iterations):
        total_loss, n_loss = (0, 0)

        hidden = model.init_hidden(batch_size=batch_size)
        batches_to_train = len(input_batches)
    #     batches_to_train = 32     # aktifkan untuk uji cepat
        for batch_i in range(batches_to_train):

            hidden = (Tensor(hidden[0].data, autograd=True), Tensor(hidden[1].data, autograd=True))

            losses = list()
            for t in range(bptt):
                input = Tensor(input_batches[batch_i][t], autograd=True)
                rnn_input = embed.forward(input=input)
                output, hidden = model.forward(input=rnn_input, hidden=hidden)

                target = Tensor(target_batches[batch_i][t], autograd=True)
                batch_loss = criterion.forward(output, target)

                if(t == 0):
                    losses.append(batch_loss)
                else:
                    losses.append(batch_loss + losses[-1])

            loss = losses[-1]

            loss.backward()
            optim.step()
            total_loss += loss.data / bptt

            epoch_loss = np.exp(total_loss / (batch_i+1))
            if(epoch_loss < min_loss):
                min_loss = epoch_loss
                print()

            log = "\r Iter:" + str(iter)
            log += " - Alpha:" + str(optim.alpha)[0:5]
            log += " - Batch "+str(batch_i+1)+"/"+str(len(input_batches))
            log += " - Min Loss:" + str(min_loss)[0:5]
            log += " - Loss:" + str(epoch_loss)
            if(batch_i == 0):
                log += " - " + generate_sample(n=70, init_char='T').replace("\n"," ")
            if(batch_i % 1 == 0):
                sys.stdout.write(log)
        optim.alpha *= 0.99

train(EPOCHS_LSTM)

Loss (dalam perplexity, 62 berarti menebak acak) turun dari 62 di awal menjadi sekitar 19,3 pada akhir epoch 0 dan sekitar 13,4 pada akhir epoch 1 (nilai terkecil 12,76). Ini jauh lebih cepat dan stabil daripada RNN di atas, yang setelah tiga epoch baru sekitar 18,6 dan sempat melonjak sampai orde 10^11. Satu epoch LSTM memakan sekitar 110 detik di mesin saya. Teks contoh di awal epoch 1 sudah berupa kata-kata (`her ther ther ...`), walau masih berulang.

### Contoh teks dari LSTM

In [ ]:
print(generate_sample(n=300, init_char='\n'))

Teks yang dihasilkan (`An will wis will wis ...`) berupa kata-kata Inggris yang berulang. Penyebabnya `argmax`: model selalu mengambil karakter paling mungkin sehingga jatuh ke loop. Dua epoch belum cukup untuk menulis seperti naskah; itu butuh ratusan epoch.

## 8. Uji tambahan: memeriksa gradien framework

Bagian ini bukan dari kode chapter. Chapter ini menekankan bahwa kebanyakan bug pada
framework tidak memunculkan error, hanya membuat jaringan tidak belajar sebagaimana
mestinya. Karena LSTM tadi berhenti dengan error sebelum saya mengganti `raise` dengan
`return`, saya ingin tahu apakah gradiennya benar.

Caranya, numerical gradient check: bangun LSTM kecil (5 karakter, 4 unit, 3 langkah),
hitung gradien dengan autograd, lalu bandingkan dengan turunan numerik
`(loss(w + eps) - loss(w - eps)) / (2 * eps)` untuk setiap elemen bobot. Sel di bawah
mencetak selisih terbesar per bobot. Selisih yang benar seharusnya sangat kecil (sekitar
1e-10).

In [ ]:
# Percobaan tambahan (bukan dari kode buku): numerical gradient check pada LSTM kecil
def grad_check(T=3, eps=1e-5):
    np.random.seed(0)
    V, H = 5, 4
    emb = Embedding(V, H)
    cell = LSTMCell(H, H, V)
    crit = CrossEntropyLoss()
    seq = [0, 1, 2, 3]

    def build_loss():
        hidden = cell.init_hidden(batch_size=1)
        loss = None
        for t in range(T):
            out, hidden = cell.forward(emb.forward(Tensor(np.array([seq[t]]), autograd=True)), hidden)
            l = crit.forward(out, Tensor(np.array([seq[t+1]]), autograd=True))
            loss = l if loss is None else loss + l
        return loss

    params = {'embed': emb.weight, 'xf': cell.xf.weight, 'xi': cell.xi.weight,
              'xo': cell.xo.weight, 'xc': cell.xc.weight, 'hf': cell.hf.weight,
              'hi': cell.hi.weight, 'ho': cell.ho.weight, 'hc': cell.hc.weight,
              'w_ho': cell.w_ho.weight}

    build_loss().backward()
    analytic = {k: p.grad.data.copy() for k, p in params.items()}

    errors = {}
    for k, p in params.items():
        worst = 0.0
        for idx in np.ndindex(*p.data.shape):
            old = p.data[idx]
            p.data[idx] = old + eps; plus = float(build_loss().data)
            p.data[idx] = old - eps; minus = float(build_loss().data)
            p.data[idx] = old
            worst = max(worst, abs(analytic[k][idx] - (plus - minus) / (2 * eps)))
        errors[k] = worst
    return errors

for k, v in grad_check().items():
    print(k.ljust(6), '%.1e' % v)

Hasilnya mengejutkan: hanya `w_ho` yang cocok dengan turunan numerik (7e-11). Bobot LSTM dan embedding menyimpang: `xf` 1,9e-3, `xi` 4,8e-3, `xo` 3,6e-3, `xc` 1,7e-2, `hf` 3,1e-2, `hi` 9,1e-2, `ho` 7,0e-2, `hc` 1,5e-1, `embed` 1,6e-1. Artinya `return` hanya menyembunyikan error: gradien yang sampai ke lapisan di bawah output tidak lengkap, karena `backward` pada framework ini tidak membawa asal gradien (`grad_origin`) dengan benar saat tensor dipakai berulang oleh banyak gate. Saya sudah menguji versi framework yang diperbaiki di luar notebook: galatnya turun menjadi sekitar 1e-11, tetapi dengan alpha 0,05 pada loss yang dijumlahkan training-nya menjadi NaN. Jadi hasil training di atas tetap berasal dari gradien yang sebagian tidak tepat; loss-nya tetap turun, tetapi ini pelajaran bahwa framework buatan sendiri perlu diuji, bukan dipercaya karena lossnya turun.

## Key Ideas — Chapter 14

- Language model karakter memprediksi karakter berikutnya; vocabulary-nya adalah karakter dalam dataset.
- Backpropagation melalui seluruh teks tidak mungkin, jadi dipakai truncated backpropagation dengan panjang `bptt` yang bisa diatur (umumnya 16 sampai 64).
- Truncated backpropagation membatasi seberapa jauh jaringan bisa belajar mengingat, dan membuat logika mini-batch lebih rumit.
- Pada RNN vanilla, forward propagation stabil karena nonlinearitas, tetapi backpropagation cenderung menghasilkan gradien yang menghilang (sigmoid) atau meledak (relu).
- LSTM membentuk cell state dengan menyalin cell sebelumnya lalu menambah atau menghapus informasi lewat gate (`f`, `i`, `o`), tanpa perkalian matriks atau nonlinearitas pada cell itu sendiri.
- Informasi jarak jauh dibawa oleh `c`; `h` hanya tafsiran lokal dari `c` untuk prediksi dan gate berikutnya.
- Berkat framework dari Chapter 13, mengganti RNN dengan LSTM hanya mengubah beberapa baris.
- Bug pada framework sering tidak menimbulkan error, sehingga gradien perlu diperiksa dengan numerical gradient check.

## Kesimpulan

Chapter 14 membawa RNN ke tugas yang lebih sulit, yaitu memodelkan karakter pada karya
Shakespeare, dengan truncated backpropagation agar training tetap mungkin. Contoh kecil
memperlihatkan mengapa RNN vanilla sulit dilatih: gradien menghilang pada sigmoid dan
meledak pada relu. LSTM mengatasinya dengan cell state yang diperbarui lewat salinan
dan gate. Training model karakter dengan NumPy murni sangat lambat, jadi notebook ini
hanya menjalankan beberapa epoch. Chapter berikutnya membahas federated learning.